## __API Call — TCGA-BRCA Case & File Metadata__

Pulls case-level clinical/demographic metadata and file-level RNA-seq metadata from the GDC API, checks the resulting cohort for duplicate samples and tumor/control overlap, and exports the final tumor and control UUID lists used by the rest of the pipeline.

### __Imports & API endpoints__

In [13]:
import requests, os, json, pandas as pd 
from dotenv import load_dotenv
load_dotenv("project.env")

os.makedirs(os.getenv("Output_path"), exist_ok=True)

cases_API_endpt = "https://api.gdc.cancer.gov/cases"
files_API_endpt = "https://api.gdc.cancer.gov/files"

### __Cases API — clinical, demographic & follow-up metadata__

In [14]:
filters = {
    "op": "in", 
    "content": {
        "field": "cases.project.project_id",
        "value": ["TCGA-BRCA"]
    }
}

fields = [
    # diagnoses
    "diagnoses.ajcc_pathologic_stage",
    "diagnoses.morphology",
    "follow_ups.days_to_follow_up",
    "diagnoses.primary_diagnosis",

    # Treatments 
    "diagnoses.treatments.treatment_type", 
    "diagnoses.age_at_diagnosis",
    "diagnoses.treatments.days_to_treatment_start",
    "diagnoses.treatments.days_to_treatment_end",

    # Demographics 
    "demographic.days_to_death",
    "demographic.vital_status",
    "demographic.ethnicity",
    "demographic.race",
    "demographic.sex_at_birth", 

    # file
    "case_id",
    "files.file_id",
    "samples.sample_type"
]

field_BRCA = ",".join(fields)

params = {
    "filters": json.dumps(filters),
    "fields": field_BRCA,
    "expand": "follow_ups", 
    "format": "JSON",
    "size": "10000"
}

response_cases = requests.get(cases_API_endpt, params =  params)
metadata_cases = response_cases.json()

with open(os.path.join(os.getenv("Output_path"),"data_cases.json"), "w") as f:
    json.dump(metadata_cases, f, indent = 4)

### __Files API — RNA-seq file metadata__

Download the RNA seq files from the gathered IDs.

Filtered to `Primary Tumor` and `Solid Tissue Normal` samples only (Metastatic samples excluded at the API level).

In [15]:
filters = {
    "op": "and",
    "content": [
        {"op": "in",
         "content": {
            "field": "cases.project.project_id",
            "value": ["TCGA-BRCA"]    
         }
    },
        {"op": "in",
        "content": {
            "field": "data_type",
            "value": ["Gene Expression Quantification"]
        }  
    },
        {"op": "in",
        "content": {
            "field": "experimental_strategy",
            "value": ["RNA-Seq"]
        }
    },
        {"op": "in",
         "content": {
             "field": "analysis.workflow_type", 
             "value": ["STAR - Counts"]
         }
    },
        {"op": "in",
         "content": {
             "field": "access",
             "value": ["open"]
         }
    }, 
        {"op": "in",
         "content": {
             "field": "cases.samples.sample_type", 
             "value" : ["Primary Tumor", "Solid Tissue Normal"]
         }
    }
]
}

fields = [
    "file_id",
    "cases.case_id",
    "cases.samples.sample_type",
    "file_size"
]

field_BRCA = ",".join(fields)

params = {
    "filters": json.dumps(filters),
    "fields": field_BRCA,
    "format": "JSON",
    "size": "10000"
}

response_files = requests.get(files_API_endpt, params =  params)
data_files = response_files.json()

with open(os.path.join(os.getenv("Output_path"),"data_files.json"), "w") as f:
    json.dump(data_files, f, indent = 4)

### __Leakage and duplicated files check-up__

### Gather the list IDs for download

In [16]:
files_id = pd.json_normalize(
    data = data_files["data"]["hits"], 
    record_path=["cases", "samples"], 
    meta = ["file_id", "file_size",
            ["cases", "case_id"]], 
    errors='ignore'
).set_index("file_id")

### Check for duplication within the files and cases ids

In [17]:
print(f"n0 of duplicated files (RNA-seq data level): {files_id.index.duplicated().sum()}")
print(f"n0 of duplicated cases (patients level): {files_id["cases.case_id"].duplicated().sum()}")

n0 of duplicated files (RNA-seq data level): 0
n0 of duplicated cases (patients level): 129


### Split the files between Control and Tumeric

In [18]:
Tumor_UUID = files_id[files_id["sample_type"] == "Primary Tumor"].drop_duplicates(subset="cases.case_id", keep="first")
## this drop_duplicates was placed because of 11 patients having 2+ files to their name
Control_UUID = files_id[files_id["sample_type"] == "Solid Tissue Normal"]

### Ensuring no duplicates in our patients within the tumor group

In [19]:
tumor_case_ids = Tumor_UUID["cases.case_id"]
duplicated_cases_ids = tumor_case_ids[tumor_case_ids.duplicated(keep=False)]
print(f"total n0 tumor files: {len(tumor_case_ids)}")
print(f"n0 of patients with >1 Primary Tumor sample: {duplicated_cases_ids.nunique()}")
print(duplicated_cases_ids.unique())

total n0 tumor files: 1095
n0 of patients with >1 Primary Tumor sample: 0
[]


### Checking for a potential Tumeric/Control overlap

In [20]:
control_cases_set = set(Control_UUID["cases.case_id"])
tumor_cases_set = set(Tumor_UUID["cases.case_id"])
overlap = control_cases_set & tumor_cases_set

print(f"control: {len(control_cases_set)}, tumor: {len(tumor_cases_set)}")
print(f"overlap: {len(overlap)}")
print(f"unique to control: {len(control_cases_set - tumor_cases_set)}")

control: 113, tumor: 1095
overlap: 113
unique to control: 0


Final cohort sizes:
- Control: 113 values
- Tumeric: 1095 values

### __Export UUID lists__

### Creating the UUID lists

In [21]:
with open(os.path.join(os.getenv("Output_path"),"Tumor_UUID_download.txt"), "w") as f:
    for i in Tumor_UUID.index:
        f.write("%s\n" % i)


with open(os.path.join(os.getenv("Output_path"),"Control_UUID_download.txt"), "w") as f:
    for i in Control_UUID.index:
        f.write("%s\n" % i)

### Combining the two for the metadata parsing

In [22]:
control = pd.read_table(os.path.join(os.getenv("Output_path"),"Control_UUID_download.txt"), header=None)
control.columns = ["UUID"]
tumor = pd.read_table(os.path.join(os.getenv("Output_path"),"Tumor_UUID_download.txt"), header=None)
tumor.columns = ["UUID"]

In [23]:
full_UUID_list = pd.concat([control, tumor], ignore_index=True).sort_index()
full_UUID_list.to_csv(os.path.join(os.getenv("Output_path"), 'full_UUID_list.txt'), sep='\t', index=False, header=True)